## 1️⃣ Import Libraries
Import libraries required for automated cell type annotation
using CellTypist with the Adult Human Kidney reference model.

In [ ]:
import scanpy as sc
import celltypist
from celltypist import models
import pandas as pd
import matplotlib.pyplot as plt
import os

sc.set_figure_params(dpi=100, facecolor="white")

## 2️⃣ Load Processed Data
Load the fully processed AnnData object from Script 03 containing
normalized expression, Harmony embedding, UMAP coordinates,
and Leiden cluster labels — ready for cell type annotation.

In [ ]:
adata = sc.read_h5ad("../data/processed/GSE279086_processed.h5ad")
print(adata)
print(f"\nCells    : {adata.shape[0]:,}")
print(f"Genes    : {adata.shape[1]:,}")
print(f"Clusters : {adata.obs['leiden'].nunique()}")

## 3️⃣ Load CellTypist Model
Load the Adult Human Kidney CellTypist model trained on human kidney
single-cell data. This model identifies all major kidney cell types
including proximal tubule cells, podocytes, endothelial cells,
distal tubule cells, and immune populations.

In [ ]:
# Load local Adult Human Kidney model
model = models.Model.load(model="../Adult_Human_Kidney.pkl")

print("✅ Model loaded successfully")
print(f"\nNumber of cell types: {len(model.cell_types)}")
print(f"\nCell types in model:")
for ct in sorted(model.cell_types):
    print(f"  {ct}")

## 4️⃣ Run CellTypist Annotation
Annotate each cell with its predicted cell type using the Adult Human
Kidney model. We use majority voting which assigns cell type labels
at the cluster level rather than individual cells, producing more
robust and consistent annotations.

### Why Majority Voting?
- Individual cell predictions can be noisy
- Majority voting pools predictions within each Leiden cluster
- The most common prediction in a cluster becomes the cluster label
- Produces cleaner, more biologically meaningful annotations

In [ ]:
# Run CellTypist with majority voting
predictions = celltypist.annotate(
    adata,
    model=model,
    majority_voting=True,
    over_clustering='leiden'
)

# Add predictions to adata
adata = predictions.to_adata()

print("✅ CellTypist annotation complete")
print(f"\nColumns added to obs:")
print([c for c in adata.obs.columns if c not in 
       ['sample', 'disease', 'GSM', 'tissue',
        'n_genes_by_counts', 'total_counts',
        'total_counts_mt', 'pct_counts_mt',
        'total_counts_ribo', 'pct_counts_ribo',
        'leiden']])

## 5️⃣ Inspect Annotation Results
Examine the cell type labels assigned by CellTypist and summarize
the distribution of cell types across the dataset and by disease
condition.

In [ ]:
print("Cell type distribution (majority voting):")
print(adata.obs['majority_voting'].value_counts())

print(f"\nUnique cell types identified: {adata.obs['majority_voting'].nunique()}")

print(f"\nMean confidence score: {adata.obs['conf_score'].mean():.3f}")
print(f"Min confidence score : {adata.obs['conf_score'].min():.3f}")
print(f"Max confidence score : {adata.obs['conf_score'].max():.3f}")

## 6️⃣ Visualize Cell Type Annotations on UMAP
Plot the UMAP embedding colored by CellTypist majority voting labels
to visualize the spatial distribution of cell types and compare
their distribution between Type 1 Diabetes and Lean Control conditions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Plot 1 — by cell type
sc.pl.umap(
    adata,
    color='majority_voting',
    title='UMAP — Cell Types (CellTypist)',
    legend_loc='right margin',
    legend_fontsize=8,
    ax=axes[0],
    show=False
)

# Plot 2 — cell type proportions by disease
ct_props = (
    adata.obs.groupby(['disease', 'majority_voting'], observed=True)
    .size()
    .reset_index(name='count')
)
ct_props['proportion'] = ct_props.groupby('disease', observed=True)['count']\
    .transform(lambda x: x / x.sum())

ct_pivot = ct_props.pivot(
    index='majority_voting',
    columns='disease',
    values='proportion'
).fillna(0)

ct_pivot.plot(kind='bar', ax=axes[1], colormap='Set2', width=0.8)
axes[1].set_title('Cell Type Proportions by Disease', fontsize=14, fontweight='bold')
axes[1].set_xlabel("")
axes[1].set_ylabel("Proportion")
axes[1].tick_params(axis='x', rotation=90, labelsize=8)
axes[1].legend(title='Disease', fontsize=8)

# Plot 3 — confidence score
sc.pl.umap(
    adata,
    color='conf_score',
    title='UMAP — CellTypist Confidence Score',
    color_map='RdYlGn',
    ax=axes[2],
    show=False
)

plt.tight_layout()
plt.savefig("../figures/UMAP_celltypes.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/UMAP_celltypes.png")

## 7️⃣ Save Annotated AnnData Object
Save the fully annotated AnnData object containing CellTypist
cell type labels to disk. This object is the final annotated
dataset ready for differential expression analysis in Script 05.

In [12]:
import os

out_path = "../data/processed/GSE279086_annotated.h5ad"
adata.write_h5ad(out_path)

size_mb = os.path.getsize(out_path) / (1024 * 1024)

print(f"✅ Saved: {out_path}")
print(f"   Cells      : {adata.shape[0]:,}")
print(f"   Genes      : {adata.shape[1]:,}")
print(f"   Cell types : {adata.obs['majority_voting'].nunique()}")
print(f"   Size       : {size_mb:.1f} MB")
print(f"\nFinal adata summary:")
print(adata)

✅ Saved: ../data/processed/GSE279086_annotated.h5ad
   Cells      : 29,287
   Genes      : 36,601
   Cell types : 18
   Size       : 935.7 MB

Final adata summary:
AnnData object with n_obs × n_vars = 29287 × 36601
    obs: 'sample', 'disease', 'GSM', 'tissue', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'leiden', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'disease_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'umap', 'majority_voting_colors'
    obsm: 'X_harmony', 'X_pca', 'X_pca_30', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
